# Transformer模型压缩示例

本notebook演示如何使用我们的框架进行Transformer模型压缩。

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
from torch.utils.data import DataLoader

from src.models.transformer import create_base_model
from src.compression.techniques import create_compressed_model
from src.utils.helpers import (
    create_dummy_dataset,
    SimpleTokenizer,
    TextDataset,
    evaluate_model,
    print_model_summary
)

## 1. 准备数据

In [ ]:
# 创建虚拟数据集
print("创建数据集...")
train_texts, train_labels = create_dummy_dataset(num_samples=1000, num_classes=2)
test_texts, test_labels = create_dummy_dataset(num_samples=200, num_classes=2)

# 构建词汇表
tokenizer = SimpleTokenizer(vocab_size=5000)
tokenizer.build_vocab(train_texts)
vocab_size = len(tokenizer.word2idx)

print(f"词汇表大小: {vocab_size}")
print(f"训练样本: {len(train_texts)}")
print(f"测试样本: {len(test_texts)}")

In [ ]:
# 创建数据加载器
train_dataset = TextDataset(train_texts, train_labels, tokenizer, max_len=64)
test_dataset = TextDataset(test_texts, test_labels, tokenizer, max_len=64)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## 2. 创建基础模型

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 创建模型
base_model = create_base_model(vocab_size, num_classes=2)
base_model = base_model.to(device)

print_model_summary(base_model, "基础Transformer模型")

## 3. 评估基础模型

（注意：未训练的模型准确率会很低）

In [ ]:
base_metrics = evaluate_model(base_model, test_loader, device)
print(f"基础模型准确率: {base_metrics['accuracy']:.2f}%")
print(f"推理吞吐量: {base_metrics['throughput']:.1f} 样本/秒")

## 4. 创建压缩模型

In [ ]:
# 配置压缩参数
compression_config = {
    'prune_layers': [4, 5],  # 剪枝第4和第5层
    'quantize': True,         # 启用量化
    'quantize_bits': 8        # 8-bit量化
}

# 创建压缩模型
compressed_model = create_compressed_model(base_model, compression_config)
compressed_model = compressed_model.to(device)

print_model_summary(compressed_model, "压缩模型")

## 5. 评估压缩模型

In [ ]:
compressed_metrics = evaluate_model(compressed_model, test_loader, device)
print(f"压缩模型准确率: {compressed_metrics['accuracy']:.2f}%")
print(f"推理吞吐量: {compressed_metrics['throughput']:.1f} 样本/秒")

## 6. 对比分析

In [ ]:
import matplotlib.pyplot as plt

# 计算统计信息
base_params = sum(p.numel() for p in base_model.parameters())
compressed_params = sum(p.numel() for p in compressed_model.parameters())

compression_ratio = (1 - compressed_params / base_params) * 100
speedup = compressed_metrics['throughput'] / base_metrics['throughput']
acc_drop = base_metrics['accuracy'] - compressed_metrics['accuracy']

print("\n" + "="*60)
print("压缩效果总结")
print("="*60)
print(f"参数量: {base_params:,} → {compressed_params:,}")
print(f"压缩比: {compression_ratio:.1f}%")
print(f"准确率: {base_metrics['accuracy']:.2f}% → {compressed_metrics['accuracy']:.2f}%")
print(f"准确率下降: {acc_drop:.2f}%")
print(f"推理加速: {speedup:.2f}×")
print("="*60)

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 参数量对比
axes[0].bar(['Base', 'Compressed'], [base_params/1e6, compressed_params/1e6])
axes[0].set_ylabel('Parameters (M)')
axes[0].set_title('Parameter Count')

# 准确率对比
axes[1].bar(['Base', 'Compressed'], [base_metrics['accuracy'], compressed_metrics['accuracy']])
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Comparison')

# 吞吐量对比
axes[2].bar(['Base', 'Compressed'], [base_metrics['throughput'], compressed_metrics['throughput']])
axes[2].set_ylabel('Throughput (samples/s)')
axes[2].set_title('Inference Throughput')

plt.tight_layout()
plt.savefig('compression_comparison.png', dpi=150)
plt.show()

print("图表已保存为 compression_comparison.png")

## 7. 结论

本示例展示了如何使用我们的框架对Transformer模型进行压缩。通过结合剪枝和量化技术，我们可以显著减小模型大小并提高推理速度，同时保持可接受的准确率。

在实际应用中，还可以:
1. 使用知识蒸馏进一步提升压缩模型的性能
2. 调整压缩参数以获得更好的准确率-效率平衡
3. 在真实数据集上训练和评估模型